# Ôn Tập Thi Thường Xuyên: Softmax Regression (Phân Loại Đa Lớp)
> **Phong cách luyện tập sát đề thi máy trường:** Có **1 Khung tư duy chung cho mọi bài**, và mỗi bài đều **chia nhỏ thành từng hàm nối tiếp nhau** để bạn điền vào `# TODO` và chấm test case từng hàm một.

---

## KHUNG GHI NHỚ 7 BƯỚC CHO MỌI BÀI SOFTMAX (HỌC THUỘC KHUNG NÀY LÀ LÀM ĐƯỢC MỌI ĐỀ)

Bất kể đề cho dữ liệu gì (2D Gauss, Iris chữ, Glass nhãn `[1,2,3,5,6,7]`, hay ảnh MNIST 784 chiều) và bất kể **không dùng thư viện** hay **được dùng thư viện**, quy trình giải luôn đi đúng **7 bước** sau:

```text
[1] LÀM SẠCH & MÃ HÓA NHÃN   : Tách X, y; đưa nhãn bất kỳ về 0..C-1 (lưu lại classes để map ngược)
[2] CHIA TRAIN / TEST        : Chia TRƯỚC khi scale (stratify=y để giữ tỷ lệ các lớp)
[3] CHUẨN HÓA ĐẶC TRƯNG      : fit(mean, std) CHỈ trên X_train -> transform cả Train & Test (std==0 -> 1)
[4] CHỌN QUY ƯỚC MA TRẬN     : Hàng (N, d) dùng axis=1  |  Cột (d+1, N) dùng axis=0
[5] TÍNH TOÁN / HUẤN LUYỆN   :
    - Không thư viện: one_hot -> logits Z -> softmax_stable -> loss (clip) -> grad (P - Y) -> lặp GD/SGD
    - Có thư viện   : Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(...))])
[6] DỰ ĐOÁN                  : predict_proba -> argmax -> classes[idx] (nếu nhãn gốc không phải 0..C-1)
[7] ĐÁNH GIÁ                 : accuracy, confusion_matrix, precision/recall/f1 (macro, zero_division=0)
```

---

### Khung Code Xương Sống (2 Chế Độ Để Nhớ Nhanh Trong 2 Phút)

#### Chế độ 1: KHÔNG dùng thư viện ML (Pure NumPy — Quy ước hàng `(N, d)`)
```python
# 1. Mã hóa nhãn + One-hot (tránh bẫy nhãn chữ hoặc nhảy cóc như 1,2,3,5,6,7)
classes, y_idx = np.unique(np.asarray(y).ravel(), return_inverse=True)
Y = np.zeros((len(y_idx), len(classes))); Y[np.arange(len(y_idx)), y_idx] = 1.0

# 2. Scale không rò rỉ (chỉ fit trên train, chống std == 0)
mean, std = X_tr.mean(axis=0), X_tr.std(axis=0)
std = np.where(std == 0, 1.0, std)
X_tr_s, X_te_s = (X_tr - mean) / std, (X_te - mean) / std

# 3. Softmax ổn định + Loss + Gradient (lắp vào vòng lặp epochs/mini-batch)
def softmax(Z):
    ax = 0 if Z.ndim == 1 else 1
    e = np.exp(Z - np.max(Z, axis=ax, keepdims=True))
    return e / np.sum(e, axis=ax, keepdims=True)

P = softmax(X_b @ W + b)                                        # (B, C)
loss = -np.sum(Y_b * np.log(np.clip(P, 1e-15, 1.0))) / len(X_b) # scalar
dW = X_b.T @ (P - Y_b) / len(X_b) + l2_reg * W                  # (d, C)
db = np.mean(P - Y_b, axis=0)                                   # (C,)

# 4. Dự đoán
y_pred = classes[np.argmax(softmax(X_te_s @ W + b), axis=1)]
```

#### Chế độ 2: ĐƯỢC dùng thư viện (`scikit-learn`)
```python
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(solver="lbfgs", max_iter=1000))])
pipe.fit(X_tr, y_tr)
y_pred, y_proba = pipe.predict(X_te), pipe.predict_proba(X_te)
```

---

### Bảng Đối Chiếu 2 Quy Ước Ma Trận & Top Bẫy Test Case

| Thành phần | Quy ước Hàng `(N, d)` *(phổ biến)* | Quy ước Cột `(d+1, N)` *(code mẫu Tuần 4)* |
|---|---|---|
| Input `X` | `(N, d)` | `X_bar`: `(d+1, N)` *(hàng 0 toàn 1)* |
| Trọng số `W` | `W (d, C)`, `b (C,)` | `W (d+1, C)` *(hàng 0 là bias)* |
| Logits `Z` | `X @ W + b` → `(N, C)` | `W.T @ X_bar` → `(C, N)` |
| **Softmax `axis`** | **`axis=1, keepdims=True`** | **`axis=0, keepdims=True`** |
| **Argmax `pred`** | **`np.argmax(P, axis=1)`** | **`np.argmax(A, axis=0)`** |
| Gradient cập nhật | `W -= lr * (X.T @ (P - Y) / N)` | `W_new = W[-1] + eta * (xi @ (yi - ai).T)` |

**5 bẫy test case chết người:**
1. Quên trừ `max` → `exp(1000)` ra `NaN`.
2. Quên `keepdims=True` → lỗi broadcasting `(N, C) - (N,)`.
3. Quên `np.clip(P, 1e-15, 1.0)` trước `np.log` → `log(0)` ra `NaN`.
4. Nhãn không phải `0..C-1` (`glass.csv`: `1,2,3,5,6,7` hoặc nhãn chữ) → `IndexError`.
5. Truyền `multi_class='multinomial'` trên `sklearn >= 1.8` → `TypeError` (bỏ `multi_class`, chỉ cần `solver='lbfgs'` là tự động đa lớp!).


In [ ]:
# === CELL SETUP: CHẠY CELL NÀY TRƯỚC TIÊN ===
import math
import gzip
import struct
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    log_loss,
)

def data_path(filename: str) -> Path:
    for p in [Path.cwd() / "week4" / filename, Path.cwd() / filename, Path(r"D:\MachineLearning\week4") / filename]:
        if p.exists():
            return p
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {filename}")

def _collect_code_names(code_obj):
    names = set(code_obj.co_names)
    for c in code_obj.co_consts:
        if hasattr(c, "co_names"):
            names.update(_collect_code_names(c))
    return names

def assert_no_ml_library(
    fn,
    forbidden=(
        "sklearn", "scipy", "torch", "tensorflow", "keras",
        "LogisticRegression", "StandardScaler", "train_test_split",
        "accuracy_score", "confusion_matrix", "precision_score",
        "recall_score", "f1_score", "log_loss", "coo_matrix",
    ),
):
    used = _collect_code_names(fn.__code__)
    for lib in forbidden:
        assert lib not in used, f"Hàm {fn.__name__} thuộc phần KHÔNG DÙNG THƯ VIỆN nên không được dùng '{lib}'!"

print("✅ Setup xong! NumPy:", np.__version__, "| Pandas:", pd.__version__)


---
# DẠNG 1: KHÔNG DÙNG THƯ VIỆN ML — CHIA HÀM NHỎ THEO KHUNG 7 BƯỚC (QUY ƯỚC HÀNG `(N, d)`)
*(Tuyệt đối KHÔNG dùng `sklearn` hay `scipy`. Mỗi hàm chỉ 2–10 dòng; hàm sau gọi lại hàm trước đúng như cấu trúc đề thi tự động.)*

### Phần 1A — Khởi động Pure Python (`math` only, không dùng cả `numpy`)
- `py_softmax(logits)`: nhận `list[float]`, trả về `list[float]` ổn định số học.
- `py_cross_entropy(probs, true_idx)`: nhận `probs: list[float]`, `true_idx: int`, trả về `-math.log(max(probs[true_idx], 1e-15))`.
- `py_predict_one(x, W, b)`: dùng `py_softmax` để trả về `(pred_class, probs)` cho 1 mẫu `x` (`list` dài `d`), `W` (`d x C`), `b` (dài `C`).


In [ ]:
# PHẦN 1A: Điền code Pure Python (KHÔNG dùng numpy, sklearn)
def py_softmax(logits):
    # TODO: Trả về list[float] xác suất Softmax ổn định số học
    raise NotImplementedError

def py_cross_entropy(probs, true_idx):
    # TODO: Trả về float loss của 1 mẫu (chặn dưới 1e-15)
    raise NotImplementedError

def py_predict_one(x, W, b):
    # TODO: Tính logits rồi gọi py_softmax -> trả về (pred_class, probs)
    raise NotImplementedError


In [ ]:
# TEST PHẦN 1A
for f in (py_softmax, py_cross_entropy, py_predict_one):
    assert_no_ml_library(f, forbidden=("numpy", "np", "sklearn", "scipy"))

p_big = py_softmax([1e5, 1e5 + 1.0, 1e5 - 1.0])
p_sml = py_softmax([-1e5, -1e5 + 1.0, -1e5 - 1.0])
assert isinstance(p_big, list) and abs(sum(p_big) - 1.0) < 1e-9 and abs(p_big[1] - p_sml[1]) < 1e-9
assert math.isfinite(py_cross_entropy([1.0, 0.0, 0.0], 2))
cls1, pr1 = py_predict_one([2.0, 1.0], [[1.0, -1.0, 0.0], [0.5, 2.0, -0.5]], [0.1, -0.2, 0.3])
assert cls1 == 0 and abs(sum(pr1) - 1.0) < 1e-9
print("✅ PASS Phần 1A (Pure Python)!")


<details>
<summary><b>💡 Bấm để xem Đáp án mẫu đối chiếu (Chỉ mở sau khi tự làm)</b></summary>

```python
def py_softmax(logits):
    m = max(logits)
    exps = [math.exp(float(z) - m) for z in logits]
    s = sum(exps)
    return [e / s for e in exps]

def py_cross_entropy(probs, true_idx):
    p = max(min(float(probs[true_idx]), 1.0), 1e-15)
    return -math.log(p)

def py_predict_one(x, W, b):
    d, C = len(x), len(b)
    logits = [sum(float(x[j]) * float(W[j][c]) for j in range(d)) + float(b[c]) for c in range(C)]
    probs = py_softmax(logits)
    return max(range(C), key=lambda c: probs[c]), probs
```

</details>

---
### Phần 1B — Bộ 8 Hàm Rời Softmax Regression Bằng Pure NumPy (Hàm sau gọi hàm trước)
Mỗi bước trong Khung 7 Bước tương ứng với 1–2 hàm độc lập:
1. **`np_encode_labels(y)`**: ánh xạ nhãn bất kỳ (chữ hoặc số không liên tục) $\to$ `(y_idx, classes)`.
2. **`np_one_hot(y_idx, C)`**: biến `y_idx` `(N,)` $\to$ ma trận One-hot `(N, C)` dtype `float64` (không dùng `scipy`).
3. **`np_standardize(X_train, X_test)`**: chuẩn hóa Z-score chỉ fit trên `X_train` (xử lý `std == 0 -> 1.0`).
4. **`np_softmax(Z)`**: hỗ trợ cả 1D `(C,)` và 2D `(N, C)`, trừ `max` với `keepdims=True`.
5. **`np_cross_entropy(Y_onehot, P)`**: tính loss trung bình trên `N` mẫu (`np.clip(P, 1e-15, 1.0)`).
6. **`np_loss_and_grad(X, Y_onehot, W, b, l2_reg=0.0)`**: gọi `np_softmax` và `np_cross_entropy` để trả về `(loss, dW, db)`.
7. **`np_fit_softmax(X, y_idx, C, lr=0.1, epochs=400, batch_size=32, l2_reg=0.0, seed=42)`**: gọi `np_one_hot` và `np_loss_and_grad` trong vòng lặp mini-batch, trả về `(W, b, loss_history)`.
8. **`np_predict_proba(X, W, b)` & `np_predict(X, W, b, classes=None)`**: gọi `np_softmax`, trả về xác suất `(N, C)` và nhãn `(N,)`.
9. **`np_confusion_matrix(y_true, y_pred, labels=None)` & `np_eval_metrics(y_true, y_pred, labels=None)`**: trả về `(accuracy, precision_macro, recall_macro, f1_macro)`.


In [ ]:
# BƯỚC 1 & 3 (Hàm 1-3): Mã hóa nhãn, One-hot, Chuẩn hóa
def np_encode_labels(y):
    # TODO: Trả về (y_idx, classes) bằng np.unique(..., return_inverse=True)
    raise NotImplementedError

def np_one_hot(y_idx, C):
    # TODO: Trả về ma trận (N, C) dtype float64
    raise NotImplementedError

def np_standardize(X_train, X_test):
    # TODO: Chuẩn hóa Z-score fit trên X_train, thay std==0 thành 1.0
    raise NotImplementedError


In [ ]:
# TEST HÀM 1-3
for f in (np_encode_labels, np_one_hot, np_standardize):
    assert_no_ml_library(f)

y_idx_t, cls_t = np_encode_labels(["b", "a", "c", "a"])
assert np.array_equal(cls_t, np.array(["a", "b", "c"])) and np.array_equal(y_idx_t, np.array([1, 0, 2, 0]))
Y_oh_t = np_one_hot(y_idx_t, len(cls_t))
assert Y_oh_t.shape == (4, 3) and np.all(Y_oh_t.sum(axis=1) == 1.0)

Xtr_s_t, Xte_s_t = np_standardize(np.array([[1.0, 9.0], [3.0, 9.0]]), np.array([[2.0, 9.0]]))
assert np.allclose(Xtr_s_t, [[-1.0, 0.0], [1.0, 0.0]]) and np.allclose(Xte_s_t, [[0.0, 0.0]])
print("✅ PASS Hàm 1-3 (Encode, One-hot, Standardize)!")


In [ ]:
# BƯỚC 5 (Hàm 4-7): Softmax, Cross-Entropy, Gradient, Huấn luyện Mini-batch
def np_softmax(Z):
    # TODO: Hỗ trợ 1D (C,) và 2D (N, C), trừ max ổn định số học
    raise NotImplementedError

def np_cross_entropy(Y_onehot, P):
    # TODO: Loss trung bình trên N mẫu, nhớ np.clip(P, 1e-15, 1.0)
    raise NotImplementedError

def np_loss_and_grad(X, Y_onehot, W, b, l2_reg=0.0):
    # TODO: Gọi np_softmax và np_cross_entropy -> trả về (loss, dW, db)
    raise NotImplementedError

def np_fit_softmax(X, y_idx, C, lr=0.1, epochs=400, batch_size=32, l2_reg=0.0, seed=42):
    # TODO: Gọi np_one_hot và np_loss_and_grad -> trả về (W, b, loss_history)
    raise NotImplementedError


In [ ]:
# TEST HÀM 4-7 (Overflow + Numerical Gradient Check + Convergence)
for f in (np_softmax, np_cross_entropy, np_loss_and_grad, np_fit_softmax):
    assert_no_ml_library(f)

p1 = np_softmax(np.array([1000.0, 1001.0, 999.0]))
p2 = np_softmax(np.array([[1000.0, 1001.0], [-1000.0, -999.0]]))
assert p1.shape == (3,) and p2.shape == (2, 2) and np.isfinite(p2).all() and np.allclose(p2.sum(axis=1), 1.0)

rng_t = np.random.default_rng(7)
Xg = rng_t.normal(size=(6, 4))
Yg = np_one_hot(np.array([0, 2, 1, 0, 2, 1]), 3)
Wg = rng_t.normal(size=(4, 3))
bg = rng_t.normal(size=(3,))
_, dW_a, db_a = np_loss_and_grad(Xg, Yg, Wg, bg, l2_reg=0.05)

eps = 1e-6
Wp, Wm = Wg.copy(), Wg.copy(); Wp[2, 1] += eps; Wm[2, 1] -= eps
lp, _, _ = np_loss_and_grad(Xg, Yg, Wp, bg, l2_reg=0.05)
lm, _, _ = np_loss_and_grad(Xg, Yg, Wm, bg, l2_reg=0.05)
assert np.isclose(dW_a[2, 1], (lp - lm) / (2 * eps), atol=1e-6), "Gradient dW sai!"
print("✅ PASS Hàm 4-7 (Softmax stable & Gradient Check 1e-6)!")


In [ ]:
# BƯỚC 6 & 7 (Hàm 8-11): Dự đoán & Tự viết Ma trận nhầm lẫn / Độ đo Macro
def np_predict_proba(X, W, b):
    # TODO: Gọi np_softmax -> trả về (N, C)
    raise NotImplementedError

def np_predict(X, W, b, classes=None):
    # TODO: Gọi np_predict_proba -> argmax axis=1 -> map classes nếu có
    raise NotImplementedError

def np_confusion_matrix(y_true, y_pred, labels=None):
    # TODO: Trả về mảng int64 shape (C, C)
    raise NotImplementedError

def np_eval_metrics(y_true, y_pred, labels=None):
    # TODO: Gọi np_confusion_matrix -> trả về (accuracy, precision_macro, recall_macro, f1_macro)
    raise NotImplementedError


In [ ]:
# TEST HÀM 8-11 (Chạy cả Khung 7 Bước hoàn chỉnh trên Iris nhãn chuỗi!)
for f in (np_predict_proba, np_predict, np_confusion_matrix, np_eval_metrics):
    assert_no_ml_library(f)

df_ir1 = pd.read_csv(data_path("iris.csv"))
X_ir1, y_ir1 = df_ir1.iloc[:, :4].to_numpy(), df_ir1["variety"].to_numpy()
X1_tr, X1_te, y1_tr, y1_te = train_test_split(X_ir1, y_ir1, test_size=0.2, stratify=y_ir1, random_state=42)

# Lắp ghép 7 bước
y1_tr_idx, cls1_ir = np_encode_labels(y1_tr)
X1_tr_s, X1_te_s = np_standardize(X1_tr, X1_te)
W1, b1, hist1 = np_fit_softmax(X1_tr_s, y1_tr_idx, len(cls1_ir), lr=0.2, epochs=300, batch_size=16)
y1_pred = np_predict(X1_te_s, W1, b1, classes=cls1_ir)
acc1, p1_m, r1_m, f1_m = np_eval_metrics(y1_te, y1_pred, labels=cls1_ir)

assert np.isclose(acc1, accuracy_score(y1_te, y1_pred)) and acc1 >= 0.90
assert np.isclose(f1_m, f1_score(y1_te, y1_pred, labels=cls1_ir, average="macro", zero_division=0))
assert np.array_equal(np_confusion_matrix(y1_te, y1_pred, labels=cls1_ir), confusion_matrix(y1_te, y1_pred, labels=cls1_ir))
print(f"✅ PASS Hàm 8-11! Khung 7 bước trên Iris đạt Acc={acc1:.4f}, Macro-F1={f1_m:.4f}")


<details>
<summary><b>💡 Bấm để xem Đáp án mẫu Phần 1B (Hàm 1 → 11)</b></summary>

```python
def np_encode_labels(y):
    y_arr = np.asarray(y).ravel()
    classes, y_idx = np.unique(y_arr, return_inverse=True)
    return y_idx.astype(np.int64), classes

def np_one_hot(y_idx, C):
    y_idx = np.asarray(y_idx, dtype=np.int64).ravel()
    Y = np.zeros((len(y_idx), C), dtype=np.float64)
    Y[np.arange(len(y_idx)), y_idx] = 1.0
    return Y

def np_standardize(X_train, X_test):
    X_train = np.asarray(X_train, dtype=np.float64)
    X_test = np.asarray(X_test, dtype=np.float64)
    mean = X_train.mean(axis=0)
    std = np.where(X_train.std(axis=0) == 0, 1.0, X_train.std(axis=0))
    return (X_train - mean) / std, (X_test - mean) / std

def np_softmax(Z):
    Z = np.asarray(Z, dtype=np.float64)
    ax = 0 if Z.ndim == 1 else 1
    e = np.exp(Z - np.max(Z, axis=ax, keepdims=True))
    return e / np.sum(e, axis=ax, keepdims=True)

def np_cross_entropy(Y_onehot, P):
    return float(-np.sum(Y_onehot * np.log(np.clip(P, 1e-15, 1.0))) / Y_onehot.shape[0])

def np_loss_and_grad(X, Y_onehot, W, b, l2_reg=0.0):
    N = X.shape[0]
    P = np_softmax(X @ W + b)
    loss = np_cross_entropy(Y_onehot, P) + 0.5 * l2_reg * float(np.sum(W * W))
    diff = (P - Y_onehot) / N
    dW = X.T @ diff + l2_reg * W
    db = np.sum(diff, axis=0)
    return loss, dW, db

def np_fit_softmax(X, y_idx, C, lr=0.1, epochs=400, batch_size=32, l2_reg=0.0, seed=42):
    X = np.asarray(X, dtype=np.float64)
    N, d = X.shape
    Y = np_one_hot(y_idx, C)
    W = np.zeros((d, C), dtype=np.float64)
    b = np.zeros(C, dtype=np.float64)
    rng = np.random.default_rng(seed)
    loss_history = []
    for _ in range(epochs):
        perm = rng.permutation(N)
        for start in range(0, N, batch_size):
            idx = perm[start : start + batch_size]
            _, dW, db = np_loss_and_grad(X[idx], Y[idx], W, b, l2_reg=l2_reg)
            W -= lr * dW
            b -= lr * db
        ep_loss, _, _ = np_loss_and_grad(X, Y, W, b, l2_reg=l2_reg)
        loss_history.append(ep_loss)
    return W, b, loss_history

def np_predict_proba(X, W, b):
    return np_softmax(np.asarray(X, dtype=np.float64) @ W + b)

def np_predict(X, W, b, classes=None):
    idx = np.argmax(np_predict_proba(X, W, b), axis=1)
    return classes[idx] if classes is not None else idx

def np_confusion_matrix(y_true, y_pred, labels=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    idx_map = {lbl: i for i, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for yt, yp in zip(y_true, y_pred):
        if yt in idx_map and yp in idx_map:
            cm[idx_map[yt], idx_map[yp]] += 1
    return cm

def np_eval_metrics(y_true, y_pred, labels=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    cm = np_confusion_matrix(y_true, y_pred, labels=labels)
    acc = float(np.mean(y_true == y_pred))
    tp = np.diag(cm).astype(np.float64)
    fp = np.sum(cm, axis=0) - tp
    fn = np.sum(cm, axis=1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return acc, float(np.mean(prec)), float(np.mean(rec)), float(np.mean(f1))
```

</details>

---
# DẠNG 2: KHÔNG DÙNG THƯ VIỆN — QUY ƯỚC CỘT `(d+1, N)` THEO CODE MẪU TUẦN 4
Nếu đề thi cho sẵn ma trận chuyển vị có gắn hàng bias ở đầu (`X_bar` shape `(d+1, N)`), hãy điền 5 hàm theo **`axis=0`**:
1. `to_col_with_bias(X_row)`: nhận `(N, d)` $\to$ `(d + 1, N)` có hàng 0 toàn `1.0`.
2. `one_hot_col(y, C)`: nhận `y` (`0..C-1`) $\to$ `(C, N)`.
3. `softmax_col(Z)`: nhận `Z` `(C, N)` $\to$ softmax ổn định theo `axis=0, keepdims=True`.
4. `softmax_sgd_col(X_bar, y, W_init, eta=0.05, tol=1e-4, max_count=10000, seed=42)`: SGD từng mẫu theo công thức `W_new = W[-1] + eta * (xi @ (yi - ai).T)`, dừng sớm sau mỗi 20 bước khi `norm(W_new - W[-21]) < tol`, trả về `W_history`.
5. `predict_col(W, X_bar)`: trả về nhãn dự đoán `(N,)` theo `axis=0`.


In [ ]:
# DẠNG 2: Điền 5 hàm quy ước cột (d+1, N)
def to_col_with_bias(X_row):
    # TODO: Trả về shape (d + 1, N)
    raise NotImplementedError

def one_hot_col(y, C):
    # TODO: Trả về shape (C, N)
    raise NotImplementedError

def softmax_col(Z):
    # TODO: Softmax ổn định theo axis=0
    raise NotImplementedError

def softmax_sgd_col(X_bar, y, W_init, eta=0.05, tol=1e-4, max_count=10000, seed=42):
    # TODO: SGD từng mẫu, trả về list lịch sử W
    raise NotImplementedError

def predict_col(W, X_bar):
    # TODO: Trả về nhãn (N,)
    raise NotImplementedError


In [ ]:
# TEST DẠNG 2
for f in (to_col_with_bias, one_hot_col, softmax_col, softmax_sgd_col, predict_col):
    assert_no_ml_library(f)

Xb_t = to_col_with_bias(np.array([[1.0, 2.0], [3.0, 4.0]]))
assert Xb_t.shape == (3, 2) and np.all(Xb_t[0, :] == 1.0)
assert one_hot_col(np.array([0, 2]), 3).shape == (3, 2)
P_c = softmax_col(np.array([[1000.0, -500.0], [1001.0, -499.0], [999.0, -501.0]]))
assert P_c.shape == (3, 2) and np.isfinite(P_c).all() and np.allclose(P_c.sum(axis=0), 1.0)

rng_c = np.random.default_rng(42)
Xg_c = np.vstack([
    rng_c.multivariate_normal([2, 2], [[1, 0], [0, 1]], 150),
    rng_c.multivariate_normal([8, 3], [[1, 0], [0, 1]], 150),
    rng_c.multivariate_normal([3, 6], [[1, 0], [0, 1]], 150),
])
yg_c = np.array([0]*150 + [1]*150 + [2]*150)
W_hist_c = softmax_sgd_col(to_col_with_bias(Xg_c), yg_c, rng_c.normal(size=(3, 3)))
acc_c = np.mean(predict_col(W_hist_c[-1], to_col_with_bias(Xg_c)) == yg_c)
assert acc_c >= 0.90
print(f"✅ PASS Dạng 2 (Quy ước cột Tuần 4)! 3-Gauss Acc = {acc_c:.4f}")


<details>
<summary><b>💡 Bấm để xem Đáp án mẫu Dạng 2</b></summary>

```python
def to_col_with_bias(X_row):
    X_row = np.asarray(X_row, dtype=np.float64)
    return np.vstack([np.ones((1, X_row.shape[0]), dtype=np.float64), X_row.T])

def one_hot_col(y, C):
    y = np.asarray(y, dtype=np.int64).ravel()
    Y = np.zeros((C, len(y)), dtype=np.float64)
    Y[y, np.arange(len(y))] = 1.0
    return Y

def softmax_col(Z):
    Z = np.asarray(Z, dtype=np.float64)
    e = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return e / np.sum(e, axis=0, keepdims=True)

def softmax_sgd_col(X_bar, y, W_init, eta=0.05, tol=1e-4, max_count=10000, seed=42):
    W = [W_init.astype(np.float64).copy()]
    Y = one_hot_col(y, W_init.shape[1])
    N = X_bar.shape[1]
    rng = np.random.default_rng(seed)
    count = 0
    while count < max_count:
        for i in rng.permutation(N):
            xi, yi = X_bar[:, [i]], Y[:, [i]]
            ai = softmax_col(W[-1].T @ xi)
            W_new = W[-1] + eta * (xi @ (yi - ai).T)
            count += 1
            W.append(W_new)
            if count % 20 == 0 and len(W) > 20 and np.linalg.norm(W_new - W[-21]) < tol:
                return W
            if count >= max_count:
                break
    return W

def predict_col(W, X_bar):
    return np.argmax(softmax_col(W.T @ X_bar), axis=0)
```

</details>

---
# DẠNG 3: ĐƯỢC DÙNG THƯ VIỆN (`scikit-learn`, `pandas`, `gzip`/`struct`)
Cũng chia thành từng hàm nhỏ nối tiếp nhau theo đúng Khung 7 Bước:

### Phần 3A — Dữ liệu bảng (`iris.csv`, `glass.csv`)
1. `lib_split_data(X, y, test_size=0.25, random_state=42)`: dùng `train_test_split` có `stratify=y`.
2. `lib_build_pipeline(solver='lbfgs', C=1.0, max_iter=1000, scale=True)`: trả về `Pipeline` (`StandardScaler` + `LogisticRegression`). *(Không truyền `multi_class` để tương thích mọi bản sklearn!)*
3. `lib_fit_predict(pipe, X_train, y_train, X_test)`: fit trên train, trả về `(pipe, y_pred, y_proba)`.
4. `lib_evaluate(y_test, y_pred, y_proba, classes)`: trả về dict `{accuracy, f1_macro, log_loss, confusion_matrix}`.
5. `lib_manual_proba_from_coef(clf, X_scaled)`: tự tính `np_softmax(X_scaled @ clf.coef_.T + clf.intercept_)` để hiểu bản chất tham số của `sklearn`.

### Phần 3B — Dữ liệu ảnh MNIST `.gz`
6. `lib_read_idx_images(img_gz_path, max_samples=None)`: unpack header 16 byte `>IIII`, kiểm tra `magic == 2051`, chuẩn hóa `[0, 1]`.
7. `lib_read_idx_labels(lbl_gz_path, max_samples=None)`: unpack header 8 byte `>II`, kiểm tra `magic == 2049`.


In [ ]:
# DẠNG 3: Điền 7 hàm dùng thư viện
def lib_split_data(X, y, test_size=0.25, random_state=42):
    # TODO: Trả về (X_train, X_test, y_train, y_test) có stratify=y
    raise NotImplementedError

def lib_build_pipeline(solver="lbfgs", C=1.0, max_iter=1000, scale=True):
    # TODO: Trả về Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(...))])
    raise NotImplementedError

def lib_fit_predict(pipe, X_train, y_train, X_test):
    # TODO: Fit pipe trên train -> trả về (pipe, y_pred, y_proba)
    raise NotImplementedError

def lib_evaluate(y_test, y_pred, y_proba, classes):
    # TODO: Trả về dict {accuracy, f1_macro, log_loss, confusion_matrix}
    raise NotImplementedError

def lib_manual_proba_from_coef(clf, X_scaled):
    # TODO: Tự tính xác suất từ clf.coef_ và clf.intercept_
    raise NotImplementedError

def lib_read_idx_images(img_gz_path, max_samples=None):
    # TODO: Đọc MNIST images .gz -> shape (n, 784) float32 trong [0, 1]
    raise NotImplementedError

def lib_read_idx_labels(lbl_gz_path, max_samples=None):
    # TODO: Đọc MNIST labels .gz -> shape (n,) int64
    raise NotImplementedError


In [ ]:
# TEST DẠNG 3 (Glass + Iris + MNIST .gz)
glass_cols = ["RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe", "Type"]
df_gl = pd.read_csv(data_path("glass.csv"), header=None, names=glass_cols)
X_gl_tr, X_gl_te, y_gl_tr, y_gl_te = lib_split_data(df_gl[glass_cols[:-1]].to_numpy(), df_gl["Type"].to_numpy())
pipe_gl = lib_build_pipeline(max_iter=2000)
pipe_gl, yp_gl, ypr_gl = lib_fit_predict(pipe_gl, X_gl_tr, y_gl_tr, X_gl_te)
res_gl = lib_evaluate(y_gl_te, yp_gl, ypr_gl, pipe_gl.named_steps["clf"].classes_)
assert res_gl["accuracy"] >= 0.55 and res_gl["confusion_matrix"].shape == (6, 6)

# Kiểm chứng coef_ & intercept_
X_gl_te_s = pipe_gl.named_steps["scaler"].transform(X_gl_te)
assert np.allclose(lib_manual_proba_from_coef(pipe_gl.named_steps["clf"], X_gl_te_s), ypr_gl, atol=1e-10)

# Kiểm tra MNIST .gz
X_m_tr = lib_read_idx_images(data_path("train-images-idx3-ubyte.gz"), max_samples=2500)
y_m_tr = lib_read_idx_labels(data_path("train-labels-idx1-ubyte.gz"), max_samples=2500)
X_m_te = lib_read_idx_images(data_path("t10k-images-idx3-ubyte.gz"), max_samples=600)
y_m_te = lib_read_idx_labels(data_path("t10k-labels-idx1-ubyte.gz"), max_samples=600)
assert X_m_tr.shape == (2500, 784) and y_m_tr.shape == (2500,)
pipe_m = lib_build_pipeline(max_iter=300, scale=False)
_, yp_m, _ = lib_fit_predict(pipe_m, X_m_tr, y_m_tr, X_m_te)
acc_m = accuracy_score(y_m_te, yp_m)
assert acc_m >= 0.84
print(f"✅ PASS Dạng 3! Glass Acc={res_gl['accuracy']:.4f} | MNIST Subset Acc={acc_m:.4f}")


<details>
<summary><b>💡 Bấm để xem Đáp án mẫu Dạng 3</b></summary>

```python
def lib_split_data(X, y, test_size=0.25, random_state=42):
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

def lib_build_pipeline(solver="lbfgs", C=1.0, max_iter=1000, scale=True):
    steps = [("scaler", StandardScaler())] if scale else []
    steps.append(("clf", LogisticRegression(solver=solver, C=C, max_iter=max_iter, random_state=42)))
    return Pipeline(steps)

def lib_fit_predict(pipe, X_train, y_train, X_test):
    pipe.fit(X_train, y_train)
    return pipe, pipe.predict(X_test), pipe.predict_proba(X_test)

def lib_evaluate(y_test, y_pred, y_proba, classes):
    return {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1_macro": float(f1_score(y_test, y_pred, labels=classes, average="macro", zero_division=0)),
        "log_loss": float(log_loss(y_test, y_proba, labels=classes)),
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=classes),
    }

def lib_manual_proba_from_coef(clf, X_scaled):
    Z = np.asarray(X_scaled, dtype=np.float64) @ clf.coef_.T + clf.intercept_
    e = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

def lib_read_idx_images(img_gz_path, max_samples=None):
    with gzip.open(img_gz_path, "rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"Magic ảnh sai: {magic}")
        X = np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows * cols).astype(np.float32) / 255.0
    return X[:max_samples] if max_samples is not None else X

def lib_read_idx_labels(lbl_gz_path, max_samples=None):
    with gzip.open(lbl_gz_path, "rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"Magic nhãn sai: {magic}")
        y = np.frombuffer(f.read(), dtype=np.uint8).astype(np.int64)
    return y[:max_samples] if max_samples is not None else y
```

</details>

---
# PHẦN THI THỬ GIỐNG ĐỀ TRƯỜNG: 10 HÀM ĐIỀN CODE (10 × 1.0đ = 10.0 ĐIỂM)
Cấu trúc giống hệt bài thi chấm test case: chia thành **10 hàm nhỏ độc lập/nối tiếp** để chấm riêng từng câu.
- **Câu 1 → Câu 7 (7.0đ):** **KHÔNG** dùng `sklearn`/`scipy` (Auto-grader kiểm tra bytecode từng hàm).
- **Câu 8 → Câu 10 (3.0đ):** **Được dùng** `sklearn`, `gzip`, `struct`.

### Đặc tả 10 hàm cần điền:
**Bài 1 — Xây dựng Softmax Regression từ đầu (5.0đ):**
1. `q1_encode_and_onehot(y)` *(1.0đ)*: nhận nhãn 1D bất kỳ $\to$ trả về `(y_idx, classes, Y_onehot)` với `Y_onehot` shape `(N, C)`.
2. `q2_scale_no_leak(X_train, X_test)` *(1.0đ)*: chuẩn hóa Z-score chỉ fit trên `X_train` (xử lý `std == 0 -> 1.0`) $\to$ `(X_tr_s, X_te_s)`.
3. `q3_softmax(Z)` *(1.0đ)*: xử lý cả 1D `(C,)` và 2D `(N, C)`, ổn định số học.
4. `q4_loss_and_grad(X, Y_onehot, W, b, l2_reg=0.0)` *(1.0đ)*: dùng `q3_softmax`, trả về `(loss, dW, db)`.
5. `q5_train_and_predict(X_train, y_train, X_test, lr=0.1, epochs=400, batch_size=32, seed=42)` *(1.0đ)*: **gọi lại `q1`, `q2`, `q3`, `q4`** để huấn luyện Mini-batch GD và trả về `(y_pred, y_proba)` trên `X_test`.

**Bài 2 — Tự tính độ đo & So sánh với Thư viện / MNIST (5.0đ):**
6. `q6_confusion_matrix(y_true, y_pred, labels=None)` *(1.0đ)*: trả về ma trận `int64` `(C, C)` không dùng `sklearn`.
7. `q7_acc_and_macro_f1(y_true, y_pred, labels=None)` *(1.0đ)*: gọi `q6_confusion_matrix`, trả về `(accuracy, macro_f1)` không dùng `sklearn`.
8. `q8_sklearn_pipeline_predict(X_train, y_train, X_test, C=1.0, max_iter=1000)` *(1.0đ)*: dùng `Pipeline(StandardScaler, LogisticRegression)` $\to$ trả về `(model, y_pred, y_proba)`.
9. `q9_sklearn_eval(y_true, y_pred, y_proba, classes)` *(1.0đ)*: dùng `sklearn.metrics` trả về dict `{accuracy, f1_macro, log_loss}`.
10. `q10_load_mnist_idx(img_gz_path, lbl_gz_path, max_samples=None)` *(1.0đ)*: đọc MNIST `.gz`, kiểm tra magic `2051`/`2049`, chuẩn hóa `[0, 1]`, trả về `(X, y)`.


In [ ]:
# === PHẦN THI THỬ: ĐIỀN CODE VÀO 10 HÀM DƯỚI ĐÂY ===

# --- Câu 1..5: Xây dựng Softmax từ đầu (KHÔNG sklearn/scipy) ---
def q1_encode_and_onehot(y):
    # TODO (1.0đ): Trả về (y_idx, classes, Y_onehot)
    raise NotImplementedError

def q2_scale_no_leak(X_train, X_test):
    # TODO (1.0đ): Trả về (X_train_s, X_test_s)
    raise NotImplementedError

def q3_softmax(Z):
    # TODO (1.0đ): Hỗ trợ 1D (C,) & 2D (N, C), ổn định số học
    raise NotImplementedError

def q4_loss_and_grad(X, Y_onehot, W, b, l2_reg=0.0):
    # TODO (1.0đ): Gọi q3_softmax -> trả về (loss, dW, db)
    raise NotImplementedError

def q5_train_and_predict(X_train, y_train, X_test, lr=0.1, epochs=400, batch_size=32, seed=42):
    # TODO (1.0đ): Lắp ghép q1, q2, q3, q4 -> trả về (y_pred, y_proba)
    raise NotImplementedError

# --- Câu 6..7: Độ đo tự viết (KHÔNG sklearn/scipy) ---
def q6_confusion_matrix(y_true, y_pred, labels=None):
    # TODO (1.0đ): Trả về mảng int64 shape (C, C)
    raise NotImplementedError

def q7_acc_and_macro_f1(y_true, y_pred, labels=None):
    # TODO (1.0đ): Gọi q6_confusion_matrix -> trả về (accuracy, macro_f1)
    raise NotImplementedError

# --- Câu 8..10: Dùng thư viện sklearn & đọc MNIST .gz ---
def q8_sklearn_pipeline_predict(X_train, y_train, X_test, C=1.0, max_iter=1000):
    # TODO (1.0đ): Pipeline(StandardScaler, LogisticRegression) -> (model, y_pred, y_proba)
    raise NotImplementedError

def q9_sklearn_eval(y_true, y_pred, y_proba, classes):
    # TODO (1.0đ): Trả về dict {accuracy, f1_macro, log_loss}
    raise NotImplementedError

def q10_load_mnist_idx(img_gz_path, lbl_gz_path, max_samples=None):
    # TODO (1.0đ): Đọc ảnh & nhãn MNIST .gz -> (X, y)
    raise NotImplementedError


In [ ]:
# === AUTO-GRADER ĐỀ THI THỬ (CHẤM RIÊNG TỪNG HÀM 1.0đ / TỔNG 10.0 ĐIỂM) ===
def run_10q_autograder():
    score = 0.0
    logs = []

    def check(label, fn):
        nonlocal score
        try:
            fn()
            score += 1.0
            logs.append(f"[PASS] {label:<38} +1.0/1.0")
        except Exception as e:
            logs.append(f"[FAIL] {label:<38} +0.0/1.0  --> {type(e).__name__}: {e}")

    def t1():
        assert_no_ml_library(q1_encode_and_onehot)
        yi, cls, Yoh = q1_encode_and_onehot([5, 1, 7, 5])
        assert np.array_equal(cls, [1, 5, 7]) and np.array_equal(yi, [1, 0, 2, 1])
        assert Yoh.shape == (4, 3) and np.all(Yoh.sum(axis=1) == 1.0)

    def t2():
        assert_no_ml_library(q2_scale_no_leak)
        Xtrs, Xtes = q2_scale_no_leak(np.array([[0.0, 5.0], [2.0, 5.0]]), np.array([[1.0, 9.0]]))
        assert np.isfinite(Xtrs).all() and np.isfinite(Xtes).all()
        assert np.allclose(Xtrs[:, 0], [-1.0, 1.0]) and np.allclose(Xtes[0, 0], 0.0)

    def t3():
        assert_no_ml_library(q3_softmax)
        p1 = q3_softmax([1e6, 1e6 + 1, 1e6 - 1])
        p2 = q3_softmax(np.array([[-1e6, -1e6 + 2], [0.0, 0.0]]))
        assert p1.shape == (3,) and np.isclose(p1.sum(), 1.0)
        assert p2.shape == (2, 2) and np.isfinite(p2).all() and np.allclose(p2[1], [0.5, 0.5])

    def t4():
        assert_no_ml_library(q4_loss_and_grad)
        rng = np.random.default_rng(11)
        X = rng.normal(size=(6, 3))
        _, _, Y = q1_encode_and_onehot([0, 1, 2, 1, 0, 2])
        W, b = rng.normal(size=(3, 3)), rng.normal(size=(3,))
        loss, dW, db = q4_loss_and_grad(X, Y, W, b, l2_reg=0.1)
        assert isinstance(loss, float) and dW.shape == (3, 3) and db.shape == (3,)
        eps = 1e-6
        Wp, Wm = W.copy(), W.copy(); Wp[0, 2] += eps; Wm[0, 2] -= eps
        assert np.isclose(dW[0, 2], (q4_loss_and_grad(X, Y, Wp, b, 0.1)[0] - q4_loss_and_grad(X, Y, Wm, b, 0.1)[0]) / (2 * eps), atol=1e-5)

    def t5():
        assert_no_ml_library(q5_train_and_predict)
        rng = np.random.default_rng(99)
        X = np.vstack([
            rng.normal([-3, 0], 0.6, (35, 2)),
            rng.normal([3, 0], 0.6, (35, 2)),
            rng.normal([0, 4], 0.6, (35, 2)),
        ])
        X = np.column_stack([X, np.ones(len(X))])  # Cột hằng số std==0
        y = np.array([-2]*35 + [5]*35 + [10]*35)
        perm = rng.permutation(len(X))
        yp, yprob = q5_train_and_predict(X[perm[:75]], y[perm[:75]], X[perm[75:]])
        assert yp.shape == (30,) and yprob.shape == (30, 3) and np.allclose(yprob.sum(axis=1), 1.0)
        assert np.mean(yp == y[perm[75:]]) >= 0.90

    def t6():
        assert_no_ml_library(q6_confusion_matrix)
        cm = q6_confusion_matrix([1, 1, 3, 5], [1, 3, 3, 3], labels=[1, 3, 5])
        assert np.array_equal(cm, confusion_matrix([1, 1, 3, 5], [1, 3, 3, 3], labels=[1, 3, 5]))

    def t7():
        assert_no_ml_library(q7_acc_and_macro_f1)
        yt, yp = [1, 1, 3, 3, 5, 5], [1, 3, 3, 3, 1, 3]
        acc, mf1 = q7_acc_and_macro_f1(yt, yp, labels=[1, 3, 5])
        assert np.isclose(acc, accuracy_score(yt, yp))
        assert np.isclose(mf1, f1_score(yt, yp, labels=[1, 3, 5], average="macro", zero_division=0))

    def t8():
        ir = load_iris()
        Xtr, Xte, ytr, yte = train_test_split(ir.data, ir.target, test_size=0.2, stratify=ir.target, random_state=42)
        m, yp, ypr = q8_sklearn_pipeline_predict(Xtr, ytr, Xte)
        assert isinstance(m, Pipeline) and yp.shape == (30,) and ypr.shape == (30, 3) and accuracy_score(yte, yp) >= 0.93

    def t9():
        ir = load_iris()
        Xtr, Xte, ytr, yte = train_test_split(ir.data, ir.target, test_size=0.2, stratify=ir.target, random_state=42)
        m, yp, ypr = q8_sklearn_pipeline_predict(Xtr, ytr, Xte)
        ev = q9_sklearn_eval(yte, yp, ypr, m.named_steps["clf"].classes_)
        assert {"accuracy", "f1_macro", "log_loss"}.issubset(ev.keys()) and ev["accuracy"] >= 0.93 and ev["log_loss"] > 0

    def t10():
        X_m, y_m = q10_load_mnist_idx(data_path("t10k-images-idx3-ubyte.gz"), data_path("t10k-labels-idx1-ubyte.gz"), max_samples=500)
        assert X_m.shape == (500, 784) and y_m.shape == (500,) and 0.0 <= X_m.min() and X_m.max() <= 1.0

    for idx, (name, fn) in enumerate([
        ("Câu 1: q1_encode_and_onehot", t1),
        ("Câu 2: q2_scale_no_leak", t2),
        ("Câu 3: q3_softmax", t3),
        ("Câu 4: q4_loss_and_grad", t4),
        ("Câu 5: q5_train_and_predict", t5),
        ("Câu 6: q6_confusion_matrix", t6),
        ("Câu 7: q7_acc_and_macro_f1", t7),
        ("Câu 8: q8_sklearn_pipeline_predict", t8),
        ("Câu 9: q9_sklearn_eval", t9),
        ("Câu 10: q10_load_mnist_idx", t10),
    ], 1):
        check(name, fn)

    print("\n".join(logs))
    print("-" * 55)
    print(f"TỔNG ĐIỂM THI THỬ: {score:.1f} / 10.0")

run_10q_autograder()


<details>
<summary><b>💡 Bấm để xem Đáp án mẫu Đề thi thử (10 hàm / 10 điểm)</b></summary>

```python
def q1_encode_and_onehot(y):
    classes, y_idx = np.unique(np.asarray(y).ravel(), return_inverse=True)
    y_idx = y_idx.astype(np.int64)
    Y = np.zeros((len(y_idx), len(classes)), dtype=np.float64)
    Y[np.arange(len(y_idx)), y_idx] = 1.0
    return y_idx, classes, Y

def q2_scale_no_leak(X_train, X_test):
    X_tr = np.asarray(X_train, dtype=np.float64)
    X_te = np.asarray(X_test, dtype=np.float64)
    mean = X_tr.mean(axis=0)
    std = np.where(X_tr.std(axis=0) == 0, 1.0, X_tr.std(axis=0))
    return (X_tr - mean) / std, (X_te - mean) / std

def q3_softmax(Z):
    Z = np.asarray(Z, dtype=np.float64)
    ax = 0 if Z.ndim == 1 else 1
    e = np.exp(Z - np.max(Z, axis=ax, keepdims=True))
    return e / np.sum(e, axis=ax, keepdims=True)

def q4_loss_and_grad(X, Y_onehot, W, b, l2_reg=0.0):
    N = X.shape[0]
    P = q3_softmax(X @ W + b)
    loss = float(-np.sum(Y_onehot * np.log(np.clip(P, 1e-15, 1.0))) / N) + 0.5 * l2_reg * float(np.sum(W * W))
    diff = (P - Y_onehot) / N
    return loss, X.T @ diff + l2_reg * W, np.sum(diff, axis=0)

def q5_train_and_predict(X_train, y_train, X_test, lr=0.1, epochs=400, batch_size=32, seed=42):
    _, classes, Y_tr = q1_encode_and_onehot(y_train)
    X_tr_s, X_te_s = q2_scale_no_leak(X_train, X_test)
    N, d = X_tr_s.shape
    C = len(classes)
    W, b = np.zeros((d, C), dtype=np.float64), np.zeros(C, dtype=np.float64)
    rng = np.random.default_rng(seed)
    for _ in range(epochs):
        perm = rng.permutation(N)
        for s in range(0, N, batch_size):
            idx = perm[s : s + batch_size]
            _, dW, db = q4_loss_and_grad(X_tr_s[idx], Y_tr[idx], W, b)
            W -= lr * dW
            b -= lr * db
    y_proba = q3_softmax(X_te_s @ W + b)
    return classes[np.argmax(y_proba, axis=1)], y_proba

def q6_confusion_matrix(y_true, y_pred, labels=None):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    if labels is None:
        labels = np.unique(np.concatenate([yt, yp]))
    m = {lbl: i for i, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for a, b in zip(yt, yp):
        if a in m and b in m:
            cm[m[a], m[b]] += 1
    return cm

def q7_acc_and_macro_f1(y_true, y_pred, labels=None):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    if labels is None:
        labels = np.unique(np.concatenate([yt, yp]))
    cm = q6_confusion_matrix(yt, yp, labels=labels)
    acc = float(np.mean(yt == yp))
    tp = np.diag(cm).astype(np.float64)
    fp = np.sum(cm, axis=0) - tp
    fn = np.sum(cm, axis=1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return acc, float(np.mean(f1))

def q8_sklearn_pipeline_predict(X_train, y_train, X_test, C=1.0, max_iter=1000):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C, max_iter=max_iter, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    return pipe, pipe.predict(X_test), pipe.predict_proba(X_test)

def q9_sklearn_eval(y_true, y_pred, y_proba, classes):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, labels=classes, average="macro", zero_division=0)),
        "log_loss": float(log_loss(y_true, y_proba, labels=classes)),
    }

def q10_load_mnist_idx(img_gz_path, lbl_gz_path, max_samples=None):
    with gzip.open(img_gz_path, "rb") as fi:
        magic_i, n, r, c = struct.unpack(">IIII", fi.read(16))
        if magic_i != 2051:
            raise ValueError("Magic ảnh không hợp lệ")
        X = np.frombuffer(fi.read(), dtype=np.uint8).reshape(n, r * c).astype(np.float32) / 255.0
    with gzip.open(lbl_gz_path, "rb") as fl:
        magic_l, _ = struct.unpack(">II", fl.read(8))
        if magic_l != 2049:
            raise ValueError("Magic nhãn không hợp lệ")
        y = np.frombuffer(fl.read(), dtype=np.uint8).astype(np.int64)
    return (X[:max_samples], y[:max_samples]) if max_samples is not None else (X, y)
```

</details>